# Read NSV Registry — the base graph of the Approved Non-Standard Variable Registry

The sheet "Approved NSVs-alpha" of the pinned registry file, as published:

1. pin check: the file's SHA-256 equals its pin in `Reference_Versions.json`;
2. the sheet as a text table (`cfa.read.read_xlsx_table`: header row from the schema, every cell text, the OOXML escape `_xHHHH_` decoded); the header must equal the schema's columns;
3. the graph by the CSVW table description generated from `schema/nsv_registry.yaml` (`cfa.rule.table_to_graph`, CSVW minimal mode);
4. structural SHACL (generated by gen-shacl): must conform;
5. writes `nsv-registry.ttl` (longturtle) and its graph hash (canonical N-Triples);
6. the Excel view `NSV_Registry.xlsx`, rebuilt from the graph as written; the view's text hash must equal the source table's text hash (lossless);
7. `Provenance.json`.

The rdf-tabular cross-check of the same table is in `checks/CSVW_Crosscheck.ipynb`.

In [ ]:
import json
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(REPO_ROOT))

import pyshacl
from linkml.generators.shaclgen import ShaclGenerator
from rdflib import Graph

from cfa import SCHEMA_DIR
from cfa.graph import graph_sha256, write_longturtle
from cfa.pins import checked_source_file, table_file
from cfa.provenance import flat, provenance, write_provenance
from cfa.read import read_xlsx_table
from cfa.rule import (check_module, csvw_metadata, graph_iri, key_columns, load_module, module_source,
                      pct, table_to_graph)
from cfa.view import schema_sheet, text_sha256, view_frame, write_view

MODULE = SCHEMA_DIR / 'nsv_registry.yaml'
TABLE = 'Approved NSVs-alpha'
OUT = REPO_ROOT / 'base' / 'nsv-registry'

sv = load_module(MODULE)
check_module(sv)
SOURCE = module_source(sv)
KEYS = key_columns(sv, TABLE)

## The generated contract is current

The CSVW description and the SHACL shapes in `schema/` (written by `checks/00_Schema.ipynb`) must equal what the module generates now.

In [ ]:
CSVW_PATH = SCHEMA_DIR / 'csvw' / f'{SOURCE}.csv-metadata.json'
SHACL_PATH = SCHEMA_DIR / 'shacl' / f'{SOURCE}.shacl.ttl'
meta = csvw_metadata(sv, TABLE, f'{pct(TABLE)}.csv')
assert json.loads(CSVW_PATH.read_text(encoding='utf-8')) == meta, 'CSVW not current: run checks/00_Schema'
assert SHACL_PATH.read_text(encoding='utf-8') == ShaclGenerator(str(MODULE)).serialize(), 'SHACL not current: run checks/00_Schema'
print('contract current')

## 1–2. Pin check, the sheet as a text table

In [ ]:
src = checked_source_file(SOURCE, table_file(SOURCE, TABLE))
header_row = int(sv.get_class(TABLE).annotations['header_row'].value)
table = read_xlsx_table(src, TABLE, skip_rows=header_row - 1)
print(src.name, '| sheet', TABLE, '| rows', len(table), '| columns', len(table.columns))

## 3–4. The graph, structural SHACL

In [ ]:
g = table_to_graph(meta, table)
conforms, _, report = pyshacl.validate(g, shacl_graph=Graph().parse(SHACL_PATH))
assert conforms, report
print('triples', len(g), '| SHACL conforms')

## 5. Write the graph, its hash

In [ ]:
TTL = OUT / f'{SOURCE}.ttl'
write_longturtle(g, TTL, {'nsv': f'https://kerfors.github.io/cdisc-for-all/def/{SOURCE}/{pct(TABLE)}#'})
written = Graph().parse(TTL)
GRAPH_HASH = graph_sha256(written)
assert GRAPH_HASH == graph_sha256(g)
print(TTL.name, GRAPH_HASH)

## 6. The Excel view, lossless

In [ ]:
view = view_frame(written, sv, TABLE)
SOURCE_HASH = text_sha256(table, KEYS)
VIEW_HASH = text_sha256(view, KEYS)
assert VIEW_HASH == SOURCE_HASH, 'the view differs from the source table'
print('text hash, source = view:', VIEW_HASH)

## 7. Provenance; write the view and read it back

In [ ]:
prov = provenance(SOURCE, [src], [MODULE, CSVW_PATH, SHACL_PATH], graph_iri(SOURCE), len(written),
                  GRAPH_HASH, SOURCE_HASH, VIEW_HASH)
write_provenance(OUT / 'Provenance.json', prov)
XLSX = OUT / 'NSV_Registry.xlsx'
readme = [
    'Approved Non-Standard Variable Registry, sheet "Approved NSVs-alpha", as published.',
    f'Rebuilt from the graph {TTL.name}: columns in published order, rows sorted by the key ({", ".join(KEYS)}).',
    'Every cell is text. The Schema sheet gives each column\'s property IRI; Provenance gives the pin and the hashes.',
    'cdisc-for-all: https://github.com/kerfors/cdisc-for-all',
]
write_view(XLSX, readme, {TABLE: view}, schema_sheet(sv, [TABLE]), flat(prov))
assert text_sha256(read_xlsx_table(XLSX, TABLE), KEYS) == SOURCE_HASH
print(XLSX.name, 'read back: text hash equal')